In [1]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
%matplotlib inline

2025-05-30 00:44:26.869161: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-05-30 00:44:26.874685: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-05-30 00:44:26.893458: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1748565866.922039   41100 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1748565866.929927   41100 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1748565866.951276   41100 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linkin

In [2]:
plt.rcParams['figure.figsize'] = (10, 8) # set default figure size, 10in by 8in

np.set_printoptions(precision=4, suppress=True)

# Chapter 4: Getting Started with neural Networks: Classification and Regression

Supporting materials for:

Chollet (2021). *Deep Learning with Python*. 2nd ed. Manning Publications Co.
[Amazon](https://www.amazon.com/Learning-Python-Second-Fran%C3%A7ois-Chollet/dp/1617296864/ref=sr_1_1?crid=32NFM2SBCJVQQ)

The examples from this chapter show some basic end-to-end machine learning workflows.  We introduce data preprocessing,
basic model architecture principles, and model evaluation.

There are examples of classification and regression problems.  Make sure you understand the fundamental difference.
A classification problem is when we try to predict a label.  For example, the simpliest classification is binary
classification.  We will predict movie reviews where the class label of the review was either positive or negative,
this is a binary classification problem.  Classification can be more complex.  We have already seen
examples of multiclass classification with the MNIST training set.  In that case, we are trying to predict which of
10 digits each image belongs to.  This is an example of multiclass classification.

A regression problem then is one where we are trying to predict a continuous value.  A good example is predicting house
prices from house features, which we have also seen examples of so far in this class.



# 4.1 Classifying movie reviews: A binary classification example

Two-class classification, or binary classification, is one of the most common kinds of machine
learning problem.

## 4.1.1 The IMDB dataset

Another example of a test/example dataset.  The IMDB dataset has 50,000 reviews from the
Internet Movie DataBase.  They are evenly split into 25,000 reviews for training and
25,000 reviews for testing.  Also each is evenly split with 50% negative and 50% positive
reviews.

Keras has IMDB dataset packaged.  It has already been preprocessed, the reviews which are sequences
of words have been turned into sequences of integers, where each integer stands for a specific
word in the dictionary.  We would normally have to do a preprocessing step like this ourselves if
working on a new text based dataset.  But ML models can't use text as input, we have to convert text
into some numeric representation to be used as input to a ML model.

Load the dataset.

In [3]:
from tensorflow.keras.datasets import imdb

(train_data, train_labels), (test_data, test_labels) = imdb.load_data(num_words=10000)

17464789/17464789 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step


The argument `num_words=10000` means we keep only the top 10,000 most frequently occurring words in the
training data.

The variables `train_data` and `test_data` are lists of reviews.  Each review is a list of word indices.

`train_labels` and `test_labels` are our binary labels, 0 for *negative* reviews and 1 for *positive* reviews.

In [4]:
print(train_data.shape)
print(train_data.dtype)
print(train_data[0])

(25000,)
object
[1, 14, 22, 16, 43, 530, 973, 1622, 1385, 65, 458, 4468, 66, 3941, 4, 173, 36, 256, 5, 25, 100, 43, 838, 112, 50, 670, 2, 9, 35, 480, 284, 5, 150, 4, 172, 112, 167, 2, 336, 385, 39, 4, 172, 4536, 1111, 17, 546, 38, 13, 447, 4, 192, 50, 16, 6, 147, 2025, 19, 14, 22, 4, 1920, 4613, 469, 4, 22, 71, 87, 12, 16, 43, 530, 38, 76, 15, 13, 1247, 4, 22, 17, 515, 17, 12, 16, 626, 18, 2, 5, 62, 386, 12, 8, 316, 8, 106, 5, 4, 2223, 5244, 16, 480, 66, 3785, 33, 4, 130, 12, 16, 38, 619, 5, 25, 124, 51, 36, 135, 48, 25, 1415, 33, 6, 22, 12, 215, 28, 77, 52, 5, 14, 407, 16, 82, 2, 8, 4, 107, 117, 5952, 15, 256, 4, 2, 7, 3766, 5, 723, 36, 71, 43, 530, 476, 26, 400, 317, 46, 7, 4, 2, 1029, 13, 104, 88, 4, 381, 15, 297, 98, 32, 2071, 56, 26, 141, 6, 194, 7486, 18, 4, 226, 22, 21, 134, 476, 26, 480, 5, 144, 30, 5535, 18, 51, 36, 28, 224, 92, 25, 104, 4, 226, 65, 16, 38, 1334, 88, 12, 16, 283, 5, 16, 4472, 113, 103, 32, 15, 16, 5345, 19, 178, 32]


Because we restriced to 10,000 most frequent words, no word index should exceed 10,000

In [5]:
max([max(sequence) for sequence in train_data])

9999

An example of decoding a review back to English words.

In [6]:
# word_index is a dictionary mapping words to an integer index
word_index = imdb.get_word_index()

# reverse the mapping, mapping integer indices to the words using the word_index dict
reverse_word_index = dict(
    [(value, key) for (key, value) in word_index.items()]
)

# decode review 0.  Note that indices are offset by 3 because 0, 1 and 2 are reserved indices
# for "padding", "start of sequence" and "unknown"
decoded_review = " ".join(
    [reverse_word_index.get(i - 3, "?") for i in train_data[0]]
)

print(decoded_review)

1641221/1641221 ━━━━━━━━━━━━━━━━━━━━ 1s 1us/step
? this film was just brilliant casting location scenery story direction everyone's really suited the part they played and you could just imagine being there robert ? is an amazing actor and now the same being director ? father came from the same scottish island as myself so i loved the fact there was a real connection with this film the witty remarks throughout the film were great it was just brilliant so much that i bought the film as soon as it was released for ? and would recommend it to everyone to watch and the fly fishing was amazing really cried at the end it was so sad and you know what they say if you cry at a film it must have been good and this definitely was also ? to the two little boy's that played the ? of norman and paul they were just brilliant children are often left out of the ? list i think because the stars that play them all grown up are such a big profile for the whole film but these children are amazing and should

## 4.1.2 Preparing the data

As you may have noticed, the reviews are simple lists of integers.  The reviews are of different lengths, e.g.

In [7]:
print(len(train_data[0]))
print(len(train_data[1]))

218
189


We need to process these lists into some other format to be suitable as input for a ML algorithm, like a neural network.  You can't
directly feed in uneven lists of integers.  We have to turn the lists into tensors.  Two basic methods used:

1. Pad lists so they all have the same length. e.g. turn into tensor of shape `(samples, max_length)`.
2. *Multi-hot encode* your lists to turn them into vectors of 0s and 1s.  Since we restricted to 10,000 words this would mean we
   have 10,000 features in the tensor, where a 1 indicates that word was present (maybe multiple times) in the review, and a 0 
   indicates the word was not present.  This may result in relatively sparse representations.
   
In our following example, we use approach 2.

Encoding the integer sequences via multi-hot encoding by hand.


In [8]:
def vectorize_sequences(sequences, dimension=10000):
    """Vectorize word index sequences into a multi-hot encoding.
    The sequences is n array of regular python lists.  We hot encode
    each word we find in the list into a vector of the indicated number
    of feature dimensions, one vector for each of the word sequences /
    reviews.
    """
    # initialize a tensor/array of the correct shape, initialized to all 0's
    results = np.zeros((len(sequences), dimension))
    
    # set specific index to 1 for each word index encountered in the review
    for sequence_num, sequence in enumerate(sequences):
        for word_index in sequence:
            results[sequence_num, word_index] = 1
            
    return results

# vectorize the training and test data
x_train = vectorize_sequences(train_data)
x_test = vectorize_sequences(test_data)

Here is what the sample inputs look like now.

In [9]:
print(x_train.shape)
print(x_train[0])

(25000, 10000)
[0. 1. 1. ... 0. 0. 0.]


We also need to vectorize the labels to get them into format needed by Keras/TensorFlow.

In [10]:
print(train_labels.dtype)
print(train_labels.shape)

y_train = np.asarray(train_labels).astype("float32")
y_test = np.asarray(test_labels).astype("float32")

print(y_train.dtype)
print(y_train.shape)

int64
(25000,)
float32
(25000,)


## 4.1.3 Building your model

The input data is vectors of 10,000 features, and the output labels are scalars (1s and 0s).

A model that performs well on such a problem is a plain stack of densely connected (`Dense`) layers with
`relu` activations.  This is also known as a simple densly connected feedforward network.

Architecture decisions:

- How many layers to use
- How many (hidden) units to choose for each layer.

There are some guidelines for making good initial decisions (see chapter 5).  We start with two intermediate layers
of 16 hidden units, with a third final output layer with a single unit to perform the scalar prediction
regarding the sentiment (positive or negative) of the review.  Notice also that we use `relu` (rectified
linear unit) activation for the intermediate layers, but `sigmoid` activation for the final layer.
`sigmoid` activation is appropriate for a binary classification, as this activation squashes the output into
a value between 0 and 1.

Model definition.

In [11]:
from tensorflow import keras
from tensorflow.keras import layers

model = keras.Sequential([
    layers.Dense(16, activation="relu"),
    layers.Dense(16, activation="relu"),
    layers.Dense(1, activation="sigmoid")
])

Recall that each `Dense` layer implements the following chain of tensor operations

```
output = relu(input @ W + b)
```

Having 16 units means the weight matrix `W` will have shape `(input_dimension, 16)`, the dot product / matrix multiplication will project the input onto a 16-dimensional representation space.  The bias vector `b` has 16 values as well which are added and then
the `relu` activation is applied.

The dimensionality can be thought of as how much freedom are you allowing the model to have.  Having more units
(higher dimensional representation space) allows mode to learn more-complex representations.  But it is more computationally
expensive.  And worse, too much freedom can lead to overfitting, which is learning noise in the specific input data.  Overfitting,
or learning these unwanted patterns, leads to improvements in predicting the data you are training with, but will most likely
decrease performance on test data, which is bad.

`relu` or rectified linear unit activation is a type of simple nonlinear transformation.  It simply zeros out
negative values.

In [12]:
# relu is really the same as the max(output, 0), but we can use the tf.keras relu
from tensorflow.keras.activations import relu

linear_transform = np.linspace(-2.0, 2.0, 100)
plt.plot(linear_transform, relu(linear_transform), 'b-', label='relu activation')
plt.axis([-2, 2, -0.5, 2.0])
plt.legend()
plt.xlabel('linear transform result')
plt.ylabel('activation function output')
plt.grid();

2025-05-30 00:44:45.038015: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


The final layer uses a `sigmoid` activation function. The sigmoid activation is a common activation for
simple binary classification.  It squashes the output into a value between 0 and 1, and can thus
be interpreted as a probability estimate of the binary output.  When we need to make an actual decision,
we usually use a threshold of 0.5 for the final prediction of a 0 or 1 binary label.

In [13]:
from tensorflow.keras.activations import sigmoid

linear_transform = np.linspace(-5.0, 5.0, 100)
plt.plot(linear_transform, sigmoid(linear_transform), 'b-', label='sigmoid activation')
plt.yticks([0, 0.25, 0.5, 0.75, 1.0])
plt.legend()
plt.xlabel('linear transform result')
plt.ylabel('activation function output')
plt.grid();

### What are activation functions and why are they necessary?

Without an activation function like `relu` or `sigmoid` (also called a non-linearity), the `Dense`
layer would consist of two linear operations, a dot product and addition

```
output = input @ W + b
```

The layer could only perform **linear transformations** (also called affine transformations) of the input
data.  The **hypothesis space** of such a layer would be the set of all possible linear transformations of the
input data into a 16-dimensional space (assuming we use 16 units in the layer).

Linear transformations are limited.  Also multiple layers would not help, a sequence of linear transformations
is no more powerful than 1 linear transformation.

In order to get access to much richer hypothesis space that will benefit from deep representations, you need
a non-linearity in the function transformation.  `relu` is simple, but is a very popular example of
a non-linear activation function.

Finally we need a loss function and an optimizer.  

Its best to use `binary_crossentropy` loss for binary classification problems.  *Crossentropy* is a quantity 
from the field of information theory that measures the distance between probability distributions, or in
this case, between the ground-truth distribution and your predictions.

As for optimizer, we will use `rmsprop`, which is usually a good default choice for virtually
any problem.

Compiling the model

In [14]:
model.compile(optimizer="rmsprop",
              loss="binary_crossentropy",
              metrics=["accuracy"])

## 4.1.4 Validating your approach

You should not evalute deep learning models (or ML models in general) on the performnce on the training data.
The real question is how well models do on data it has not been trained with and has not seen before.

It is standard practice to use a validation set to monitor accuracy of a model during training.  

Here we create a validation set by hand by setting asside 10,000 samples from the original training data
of our IMDB review dataset.

Setting aside a validation set.

In [15]:
x_val = x_train[:10000]
partial_x_train = x_train[10000:]
y_val = y_train[:10000]
partial_y_train = y_train[10000:]

We will not train for 20 epochs in mini-batches of 512 samples.  At the same time we monitor loss and
accuracy on the 10,000 validation set samples we set apart.  

Training your model.

In [16]:
history = model.fit(partial_x_train,
                    partial_y_train,
                    epochs=20,
                    batch_size=512,
                    validation_data=(x_val, y_val))

2025-05-30 00:44:47.785092: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 600000000 exceeds 10% of free system memory.


Epoch 1/20
29/30 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - accuracy: 0.6906 - loss: 0.5949

2025-05-30 00:44:57.680182: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 400000000 exceeds 10% of free system memory.


30/30 ━━━━━━━━━━━━━━━━━━━━ 7s 193ms/step - accuracy: 0.6962 - loss: 0.5895 - val_accuracy: 0.8709 - val_loss: 0.3783
Epoch 2/20
30/30 ━━━━━━━━━━━━━━━━━━━━ 2s 66ms/step - accuracy: 0.8992 - loss: 0.3180 - val_accuracy: 0.8876 - val_loss: 0.3006
Epoch 3/20
30/30 ━━━━━━━━━━━━━━━━━━━━ 2s 49ms/step - accuracy: 0.9276 - loss: 0.2249 - val_accuracy: 0.8897 - val_loss: 0.2816
Epoch 4/20
30/30 ━━━━━━━━━━━━━━━━━━━━ 2s 67ms/step - accuracy: 0.9390 - loss: 0.1879 - val_accuracy: 0.8828 - val_loss: 0.2878
Epoch 5/20
30/30 ━━━━━━━━━━━━━━━━━━━━ 1s 39ms/step - accuracy: 0.9536 - loss: 0.1496 - val_accuracy: 0.8812 - val_loss: 0.3023
Epoch 6/20
30/30 ━━━━━━━━━━━━━━━━━━━━ 1s 28ms/step - accuracy: 0.9625 - loss: 0.1247 - val_accuracy: 0.8863 - val_loss: 0.2947
Epoch 7/20
30/30 ━━━━━━━━━━━━━━━━━━━━ 1s 40ms/step - accuracy: 0.9674 - loss: 0.1083 - val_accuracy: 0.8600 - val_loss: 0.4007
Epoch 8/20
30/30 ━━━━━━━━━━━━━━━━━━━━ 1s 30ms/step - accuracy: 0.9693 - loss: 0.0980 - val_accuracy: 0.8838 - val_loss: 0

Note that the call to `model.fit()` returns a `History` object.  This gives a history of the training
loss and validation metrics over the epochs of training.

In [17]:
history_dict = history.history
history_dict.keys()

dict_keys(['accuracy', 'loss', 'val_accuracy', 'val_loss'])

We can the loss and accuracy.  Further, we can plot training and validation history of both of these to make
inferences about how well the model training went.

In [18]:
# create list of epoch numbers for the plots
epochs = np.arange(1, len(history_dict['loss']) + 1)

In [19]:
# plot training and validation loss.  I don't like using line and points on same plot, so change
# to using color for the two values on plot
plt.plot(epochs, history_dict['loss'], 'r-', label='Training loss')
plt.plot(epochs, history_dict['val_loss'], 'b-', label='Validation loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.xticks(epochs)
plt.grid()
plt.legend();

In [20]:
# plot training and validation accuracy 
plt.plot(epochs, history_dict['accuracy'], 'r-', label='Training accuracy')
plt.plot(epochs, history_dict['val_accuracy'], 'b-', label='Validation accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.xticks(epochs)
plt.grid()
plt.legend();

As you can see, training loss decreases with each epoch of training, and training accuracy improves.

However, validation loss gets worse and accuracy does not improve nor is it ever as good as training accuracy.

This model performs much better on the training data than on the validation data.  

This is an example of **overfitting**.  After the 4th epoch validation loss stops decreasing.  We are
overoptimizing on the training data and end up learning representations that are specific to
the training data (noise) and don't generalize.

Can use a range of techniques to mitigate overfitting.  Simplest approach is to stop training when
validation loss starts increasing, e.g. for only 4 epochs.

Retraining a model from scratch for 4 epochs to stop before overfitting.

In [21]:
model = keras.Sequential([
            layers.Dense(16, activation="relu"),
            layers.Dense(16, activation="relu"),
            layers.Dense(1, activation="sigmoid")
])

# train the model for 4 epochs
model.compile(optimizer="rmsprop",
              loss="binary_crossentropy",
              metrics=["accuracy"])
model.fit(x_train, y_train, epochs=4, batch_size=512)


# determine results on the test data, first number is the test loss
# and second is the accuracy
results = model.evaluate(x_test, y_test)
print(results)


2025-05-30 00:45:35.072421: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 1000000000 exceeds 10% of free system memory.


Epoch 1/4
49/49 ━━━━━━━━━━━━━━━━━━━━ 5s 35ms/step - accuracy: 0.7342 - loss: 0.5565
Epoch 2/4
49/49 ━━━━━━━━━━━━━━━━━━━━ 1s 28ms/step - accuracy: 0.9050 - loss: 0.2797
Epoch 3/4
49/49 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - accuracy: 0.9229 - loss: 0.2143
Epoch 4/4
49/49 ━━━━━━━━━━━━━━━━━━━━ 2s 29ms/step - accuracy: 0.9373 - loss: 0.1829


2025-05-30 00:45:58.537738: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 1000000000 exceeds 10% of free system memory.


782/782 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.8827 - loss: 0.2903
[0.2897089123725891, 0.8839200139045715]


This naive approach achieves an accuracy of about 87%.  State-of-the-art approaches should be able to get close to 95% accuracy
on test data.

## 4.1.5 Using a trained model to generate predictions on new data

Can use `predict` method of trained model to generate predictions on new data.

In [22]:
model.predict(x_test)

 15/782 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step   

2025-05-30 00:46:06.854886: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 1000000000 exceeds 10% of free system memory.


782/782 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step


array([[0.2102],
       [0.9999],
       [0.6896],
       ...,
       [0.1045],
       [0.0724],
       [0.5489]], dtype=float32)

## 4.1.6 Further experiments

Left as an exercise for students.  See if you can improve the model.

- Try more representation layers
- Try more or fewer units in layers
- Try using `mse` loss instead of `binary_crossentropy`
- Try using `tanh` activation instead of `relu`.

## 4.1.7 Wrapping up

- Usually need to do quite a bit of preprocessing.
- Stacks of `Dense` layers with `relu` activations can solve a wide range of problems.
- In binary classification, model should end with a `Dense` layer with one unit using `sigmoid` activation.
- Should usually use `binary_crossentropy` loss function for scalar sigmoid output on binary classification problems.
- The `rmsprop` optimizer is generally good enough for most nn/deep learning problems.
- Neural networks eventually start overfitting and get increasingly worse at generalizing to unseen data.  Need to always
  fight overfitting and monitor performance.